In [1]:
import json

import rclpy
from rclpy.node import Node

from nimbro_utils.lazy import start_and_spin_node
from vlm_gist import VlmGist, test_image_path

In [2]:
class VlmGistNode(Node):
    def __init__(self, context=None):
        super().__init__("vlm_gist_node", context=context)
        self.vlm_gist = VlmGist(self, parallel_completions=None, logger_severity=20)
        self.vlm_gist.detection.set_settings(settings={'crop_masks': False})

In [3]:
node, executor, context, thread = start_and_spin_node(VlmGistNode, blocking=False)

> Starting node 'VlmGistNode'


In [4]:
image = test_image_path
# image = "https://upload.wikimedia.org/wikipedia/commons/4/46/Table.JPG"
# image ="https://upload.wikimedia.org/wikipedia/commons/1/12/Universit%C3%A4t_Bonn.jpg"

success, message, descriptions, bboxes, confidences, masks, track_ids, metadata = node.vlm_gist.get(image=image, settings={'skip_validation': True})

[INFO] [1791120662.180805361] [vlm_gist_node.vlm_gist]: Processing '1' image
[INFO] [1791120662.873492591] [vlm_gist_node.structured_description]: Generating '1' structured description
[INFO] [1791120662.875870473] [vlm_gist_node.structured_description]: Progress: {'UNPROCESSED': 1, 'TOTAL': 1} after 0.7s
[ERROR] [1791120663.311480784] [vlm_gist_node.API]: [/nimbro_api/completions_1.prompt] Failed to read API key from environment variable 'OPENROUTER_API_KEY'.
[INFO] [1791120663.326473670] [vlm_gist_node.structured_description]: Starting retry attempt '1' of '3' to generate structured description of image '0' after failure
[ERROR] [1791120663.669475215] [vlm_gist_node.API]: [/nimbro_api/completions_1.prompt] Failed to read API key from environment variable 'OPENROUTER_API_KEY'.
[INFO] [1791120663.680823661] [vlm_gist_node.structured_description]: Starting retry attempt '2' of '3' to generate structured description of image '0' after failure
[INFO] [1791120663.881591244] [vlm_gist_node.

In [5]:
from nimbro_utils.lazy import visualize_detections, show_image
visualization = visualize_detections(image=image, boxes=bboxes, masks=masks, labels=[description['object_name'] for description in descriptions], box_format="xyxy_absolute")
print(f"{image}:")
show_image(visualization, width=1000)
for description in descriptions:
    print(f"{description['object_name']}: {description['description']}")

TypeError: 'NoneType' object is not iterable

In [ ]:
node.api_director.release()

(True, "Released completions node ['/nimbro_api/completions_1'] (1).")

In [ ]:
executor.shutdown()
node.destroy_node()
context.shutdown()
del executor, node, context, thread